# Обработка данных (Feature engineering)


## Цель этапа

На этом этапе исходные таблицы приводятся к единой, воспроизводимой схеме, пригодной для обучения модели.

| Требование к feature engineering | Как оно закрывается в notebook |
|---|---|
| Обоснованность создания признаков | Добавляются временные, поведенческие и финансовые отношения с понятным смыслом |
| Преобразование числовых признаков | Исправляются строковые представления чисел, десятичные разделители и бесконечные отношения |
| Работа с категориями | Пропуски и редкие значения кодируются единообразно, правила определяются по train |
| Работа со временем | Дата наблюдения преобразуется в месяц, давность активности считается относительно даты наблюдения |
| Отбор и исключение | Автоматически удаляются только константы и точные дубликаты; спорные гипотезы оставлены как эксперименты |
| Проверка leakage | Анализируется высокая связь с target и доступность признаков на момент прогноза |

Все основные преобразования применяются одинаково к train и test.

In [1]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

RANDOM_STATE = 42

TARGET_COL = "target"
WEIGHT_COL = "w"
ID_COL = "id"
DATE_COL = "dt"

# Порог, при котором object-колонка признаётся числовой.
NUMERIC_PARSE_THRESHOLD = 0.98

# Редкие категории определяются только по train.
MIN_CATEGORY_COUNT = 10
MAX_CATEGORY_LEVELS = 200

In [2]:
!pip install -q gdown
!gdown 1f90ZWYWgibB3Bt5wASaywzI6cH231XLD
!gdown 1zBAx8y4PSGMLOgAMBbIP_iMGxPBMevUF

Downloading...
From: https://drive.google.com/uc?id=1f90ZWYWgibB3Bt5wASaywzI6cH231XLD
To: /content/train.csv
100% 99.5M/99.5M [00:01<00:00, 79.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1zBAx8y4PSGMLOgAMBbIP_iMGxPBMevUF
To: /content/test.csv
100% 94.7M/94.7M [00:01<00:00, 53.3MB/s]


## Воспроизводимость и параметры обработки


`RANDOM_STATE` фиксирует случайность в последующих экспериментах. Порог `NUMERIC_PARSE_THRESHOLD = 0.98` означает, что текстовый столбец признаётся числовым только тогда, когда не менее 98% его непустых значений корректно преобразуются в число.

In [3]:
train_raw = pd.read_csv('train.csv', decimal=',', sep=';')
test_raw = pd.read_csv('test.csv', decimal=',', sep=';')

print("Исходный train:", train_raw.shape)
print("Исходный test:", test_raw.shape)

required_train = {ID_COL, DATE_COL, TARGET_COL, WEIGHT_COL}
required_test = {ID_COL, DATE_COL}

missing_train = required_train - set(train_raw.columns)
missing_test = required_test - set(test_raw.columns)

if missing_train:
    raise ValueError(f"В train отсутствуют обязательные поля: {sorted(missing_train)}")
if missing_test:
    raise ValueError(f"В test отсутствуют обязательные поля: {sorted(missing_test)}")
if TARGET_COL in test_raw.columns or WEIGHT_COL in test_raw.columns:
    raise ValueError("В test не должно быть target или weight")

Исходный train: (76786, 224)
Исходный test: (73214, 222)


## Первичная проверка структуры train и test

До любых преобразований проверяются размеры таблиц и наличие обязательных колонок:

- `id` — идентификатор наблюдения;
- `dt` — дата наблюдения;
- `target` — прогнозируемый доход;
- `w` — вес наблюдения в WMAE.

(В test не должно быть `target` и `w`)

In [4]:
def normalize_missing_strings(series: pd.Series) -> pd.Series:
    if not pd.api.types.is_object_dtype(series) and not pd.api.types.is_string_dtype(series):
        return series

    result = series.astype("string").str.strip()
    return result.replace({
        "": pd.NA,
        "nan": pd.NA,
        "NaN": pd.NA,
        "none": pd.NA,
        "None": pd.NA,
        "null": pd.NA,
        "NULL": pd.NA
    })


def infer_numeric_object_columns(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    excluded: set[str],
    threshold: float = NUMERIC_PARSE_THRESHOLD
) -> list[str]:
    numeric_candidates = []

    common_cols = [
        c for c in train_df.columns
        if c in test_df.columns and c not in excluded
    ]

    for col in common_cols:
        if not (
            pd.api.types.is_object_dtype(train_df[col])
            or pd.api.types.is_string_dtype(train_df[col])
        ):
            continue

        train_s = normalize_missing_strings(train_df[col])
        test_s = normalize_missing_strings(test_df[col])

        train_non_missing = train_s.dropna()
        test_non_missing = test_s.dropna()

        if len(train_non_missing) == 0:
            continue

        train_parsed = pd.to_numeric(
            train_non_missing.str.replace(",", ".", regex=False),
            errors="coerce"
        )
        test_parsed = pd.to_numeric(
            test_non_missing.str.replace(",", ".", regex=False),
            errors="coerce"
        )

        train_rate = train_parsed.notna().mean()
        test_rate = test_parsed.notna().mean() if len(test_non_missing) else 1.0

        if train_rate >= threshold and test_rate >= threshold:
            numeric_candidates.append(col)

    return numeric_candidates

## Нормализация пропусков и распознавание числовых столбцов

В исходных файлах одно и то же отсутствие значения может быть записано как пустая строка, `None`, `null` или `NaN`. Эти варианты приводятся к единому `pd.NA`, чтобы модель и последующий анализ трактовали их одинаково.

Часть числовых признаков может быть прочитана как текст из-за формата выгрузки или десятичной запятой. Для таких столбцов проверяется доля значений, которые действительно преобразуются в число. Высокий порог 98% снижает риск ошибочно превратить настоящую категорию в числовой признак.

In [5]:
# Определяем смешанные object-колонки, которые фактически являются числами.

excluded_from_numeric = {
    ID_COL,
    DATE_COL,
    TARGET_COL,
    WEIGHT_COL,
    "gender",
    "adminarea",
    "city_smart_name",
    "incomeValueCategory",
    "dp_ewb_last_employment_position",
    "period_last_act_ad"
}

numeric_object_cols = infer_numeric_object_columns(
    train_raw,
    test_raw,
    excluded=excluded_from_numeric
)

print("Object-колонок, распознанных как числовые:", len(numeric_object_cols))
print(numeric_object_cols[:30])

Object-колонок, распознанных как числовые: 34
['hdb_bki_total_max_limit', 'hdb_bki_total_cc_max_limit', 'hdb_bki_total_pil_max_limit', 'hdb_bki_active_cc_max_limit', 'hdb_bki_other_active_pil_outstanding', 'hdb_bki_total_products', 'hdb_bki_total_max_overdue_sum', 'bki_total_auto_cnt', 'dp_address_unique_regions', 'hdb_bki_total_ip_max_limit', 'hdb_bki_total_cnt', 'bki_total_oth_cnt', 'bki_total_ip_max_outstand', 'hdb_bki_total_ip_cnt', 'hdb_bki_active_cc_max_outstand', 'hdb_bki_total_pil_max_overdue', 'bki_total_il_max_limit', 'bki_total_products', 'bki_active_auto_cnt', 'bki_total_max_limit', 'hdb_bki_total_auto_max_limit', 'hdb_bki_total_micro_max_overdue', 'bki_total_active_products', 'hdb_bki_total_active_products', 'hdb_bki_total_micro_cnt', 'hdb_bki_active_pil_cnt', 'hdb_bki_total_cc_max_overdue', 'hdb_bki_total_pil_last_days', 'hdb_bki_active_pil_max_limit', 'hdb_bki_total_pil_cnt']


## Исключения из автоматического распознавания чисел

Служебные поля, даты и заведомо категориальные признаки исключаются из автоматической конвертации. Например, категория дохода может внешне напоминать число, но арифметические операции над ними не имеют смысла.


In [6]:
def convert_numeric_objects(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    columns: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in columns:
        train_s = normalize_missing_strings(train_df[col])
        test_s = normalize_missing_strings(test_df[col])

        train_df[col] = pd.to_numeric(
            train_s.str.replace(",", ".", regex=False),
            errors="coerce"
        )
        test_df[col] = pd.to_numeric(
            test_s.str.replace(",", ".", regex=False),
            errors="coerce"
        )

    return train_df, test_df


train, test = convert_numeric_objects(
    train_raw,
    test_raw,
    numeric_object_cols
)

train[DATE_COL] = pd.to_datetime(train[DATE_COL], errors="raise", dayfirst=True)
test[DATE_COL] = pd.to_datetime(test[DATE_COL], errors="raise", dayfirst=True)

train = train.sort_values(DATE_COL).reset_index(drop=True)
test = test.sort_values(DATE_COL).reset_index(drop=True)

## Приведение типов и сохранение временного порядка

Выбранные текстовые столбцы преобразуются в числовые одинаковым правилом для train и test. Некорректные единичные значения становятся пропусками, а не произвольными нулями.

Поле `dt` преобразуется в настоящий тип даты, после чего данные сортируются хронологически. Это важно, потому что дальнейшая локальная валидация строится по времени: прошлые месяцы используются для обучения, следующий месяц — для проверки.

In [7]:
def add_date_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    dt = result[DATE_COL]

    result["year"] = dt.dt.year.astype("int16")
    result["month"] = dt.dt.month.astype("int8")
    result["quarter"] = dt.dt.quarter.astype("int8")
    result["day_of_week"] = dt.dt.dayofweek.astype("int8")
    result["is_month_end"] = dt.dt.is_month_end.astype("int8")
    result["month_index"] = (dt.dt.year * 12 + dt.dt.month).astype("int32")

    return result


train = add_date_features(train)
test = add_date_features(test)

## Временные признаки

Из даты наблюдения в финальной версии используется номер месяца, день недели и др. Они позволяют модели учитывать сезонность и изменение поведения клиентов во времени.



In [8]:
# Обработка period_last_act_ad

def add_activity_recency(
    df: pd.DataFrame,
    source_col: str = "period_last_act_ad"
) -> pd.DataFrame:
    result = df.copy()

    if source_col not in result.columns:
        return result

    activity_date = pd.to_datetime(
        result[source_col].astype("string"),
        errors="coerce"
    )

    result["has_ad"] = activity_date.notna().astype("int8")
    result["ad_recency_months"] = (
        (result[DATE_COL].dt.year - activity_date.dt.year) * 12
        + result[DATE_COL].dt.month
        - activity_date.dt.month
    )

    # Будущая активность относительно наблюдения — ошибка/аномалия.
    result.loc[result["ad_recency_months"] < 0, "ad_recency_months"] = np.nan
    result = result.drop(columns=[source_col])

    return result


train = add_activity_recency(train)
test = add_activity_recency(test)

## Давность последней активности

Исходное поле `period_last_act_ad` содержит дату события. Само календарное значение менее удобно для модели, поэтому оно преобразуется в два более интерпретируемых признака:

- `has_ad` — известна ли дата активности;
- `ad_recency_months` — сколько месяцев прошло от активности до даты наблюдения.

Давность считается относительно `dt` конкретной строки, а не относительно фиксированной даты. Если активность формально относится к будущему, значение заменяется на пропуск. Это защищает от временного leakage.

In [9]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")

    result = numerator / denominator.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)


def add_business_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    if {"loan_cnt", "pil"}.issubset(result.columns):
        result["active_loans"] = (
            result["loan_cnt"].fillna(0)
            + result["pil"].fillna(0)
        )

    if {"hdb_outstand_sum", "hdb_bki_total_max_limit"}.issubset(result.columns):
        result["limit_usage"] = safe_divide(
            result["hdb_outstand_sum"],
            result["hdb_bki_total_max_limit"]
        )

    food_col = "avg_by_category__amount__sum__cashflowcategory_name__produkty"
    if {food_col, "avg_6m_all"}.issubset(result.columns):
        result["food_ratio"] = safe_divide(
            result[food_col],
            result["avg_6m_all"]
        )

    if {"avg_cur_cr_turn", "avg_cur_db_turn"}.issubset(result.columns):
        result["credit_debit_turn_ratio"] = safe_divide(
            result["avg_cur_cr_turn"],
            result["avg_cur_db_turn"]
        )

    return result


train = add_business_features(train)
test = add_business_features(test)

## Предметные финансовые признаки

В этой части создаются отношения и агрегаты, которые описывают финансовое поведение клиента:

- `active_loans` — суммарный индикатор активной кредитной нагрузки;
- `limit_usage` — доля использованного кредитного лимита;
- `food_ratio` — доля расходов на продукты;
- `credit_debit_turn_ratio` — соотношение входящих и исходящих оборотов.

Отношения часто информативнее абсолютных сумм: одинаковая задолженность имеет разный смысл для клиентов с разными лимитами и оборотами. Функция `safe_divide` заменяет деление на ноль и бесконечные значения пропусками, не создавая искусственно больших чисел.

In [10]:
categorical_cols = [
    c for c in train.columns
    if c in test.columns
    and (
        pd.api.types.is_object_dtype(train[c])
        or pd.api.types.is_string_dtype(train[c])
        or pd.api.types.is_categorical_dtype(train[c])
    )
]

print("Категориальные признаки:", categorical_cols)


def fit_category_mapping(
    train_series: pd.Series,
    min_count: int = MIN_CATEGORY_COUNT,
    max_levels: int = MAX_CATEGORY_LEVELS
) -> set[str]:
    normalized = normalize_missing_strings(train_series).fillna("__MISSING__")
    counts = normalized.value_counts(dropna=False)

    kept = counts[counts >= min_count].index.astype(str).tolist()
    return set(kept[:max_levels])


category_mappings = {}

for col in categorical_cols:
    mapping = fit_category_mapping(train[col])
    category_mappings[col] = sorted(mapping)

    train_values = normalize_missing_strings(train[col]).fillna("__MISSING__").astype(str)
    test_values = normalize_missing_strings(test[col]).fillna("__MISSING__").astype(str)

    train[col] = train_values.where(train_values.isin(mapping), "__OTHER__")
    test[col] = test_values.where(test_values.isin(mapping), "__OTHER__")

Категориальные признаки: ['gender', 'adminarea', 'city_smart_name', 'dp_ewb_last_employment_position', 'addrref']


## Категориальные признаки

Категории обрабатываются отдельно от числовых признаков:

1. текст нормализуется;
2. пропуски получают отдельное значение `MISSING`;
3. редкие и неизвестные категории объединяются в `OTHER`;
4. список сохраняемых категорий рассчитывается только по train;
5. то же отображение затем применяется к test.

Такой подход уменьшает размерность, снижает переобучение на единичных значениях и не использует распределение целевой переменной. Ограничение до 200 уровней защищает модель от признаков с чрезмерной кардинальностью.

In [11]:
# # Поиск скошенных признаков

# # Берём числовые колонки
# numeric_cols = train.select_dtypes(include=['float64', 'int64']).columns

# # Считаем skewness
# skewness = train[numeric_cols].skew().sort_values(ascending=False)

# print("Топ-20 самых скошенных признаков:")
# display(skewness.head(20))

# # Фильтруем > 10
# highly_skewed = skewness[skewness > 10]
# print(f"\nСильно скошенных (skewness > 10): {len(highly_skewed)}")

## Проверенная гипотеза: логарифмирование скошенных признаков

Следующие ячейки сохранены закомментированными намеренно. В них анализировалась асимметрия числовых признаков и рассматривалось добавление `log1p`-копий для крупных положительных сумм и оборотов.

Гипотеза была содержательно обоснована: финансовые значения часто имеют тяжёлый правый хвост, а логарифмирование может сделать различия между типичными клиентами заметнее. Однако в проведённых модельных экспериментах этот вариант не улучшил итоговую WMAE и в отдельных запусках ухудшал её. Поэтому преобразование не включено в финальный датасет, но код оставлен как подтверждение проведённого эксперимента и отклонённой гипотезы.

In [12]:
# # Отбор признаков для логорифмирования

# # 1. Исключаем отношения (они уже в диапазоне 0-1 или около)
# ratio_keywords = ['ratio', 'share', 'utilization', 'usage', '_to_', 'per_']

# # 2. Исключаем признаки, где есть отрицательные значения (логарифм только для положительных)
# # 3. Берём только те, где skewness > 10 (очень сильная асимметрия) и которые являются суммами/оборотами

# log_candidates = []

# for col in skewness.index:
#     # Пропускаем, если это отношение
#     if any(kw in col.lower() for kw in ratio_keywords):
#         continue

#     # Пропускаем, если есть отрицательные значения
#     if (train[col].min() < 0):
#         continue

#     # Пропускаем, если skewness не очень высокая
#     if skewness[col] < 10:
#         continue

#     # Проверяем, что это сумма/оборот (по названию)
#     sum_keywords = ['sum', 'turn', 'amount', 'outstand', 'limit', 'balance', 'avg', 'all', 'total', 'salary', 'income']
#     if any(kw in col.lower() for kw in sum_keywords):
#         log_candidates.append({
#             'feature': col,
#             'skewness': skewness[col],
#             'min': train[col].min(),
#             'max': train[col].max(),
#             'mean': train[col].mean()
#         })

# # Сортируем по skewness и берём топ-10
# log_candidates = sorted(log_candidates, key=lambda x: x['skewness'], reverse=True)[:10]

# print(f"Найдено кандидатов для логарифмирования: {len(log_candidates)}")
# display(pd.DataFrame(log_candidates))

In [13]:
# # Логарифмирование

# # Берём топ-10 самых скошенных
# top_n = 10
# selected_for_log = [item['feature'] for item in log_candidates[:top_n]]

# print(f"Добавляем логарифмы для {len(selected_for_log)} признаков:")

# for col in selected_for_log:
#     log_name = f'log_{col}'

#     # Добавляем в train
#     train[log_name] = np.log1p(
#         train[col].clip(lower=0)  # гарантируем, что нет отрицательных
#     )

#     # Добавляем в test
#     test[log_name] = np.log1p(
#         test[col].clip(lower=0)
#     )

#     # Считаем новую skewness
#     new_skew = train[log_name].skew()

#     print(f"{log_name}")
#     print(f"Было: skewness = {skewness[col]:.2f}")
#     print(f"Стало: skewness = {new_skew:.2f}")

# print(f"\nДобавлено {len(selected_for_log)} логарифмических признаков")

In [14]:
# Проверка на утечку данных

# Проверяем признаки с высокой корреляцией с target

# Берём числовые колонки
numeric_cols = train.select_dtypes(include=['float64', 'int64']).columns
numeric_cols = [c for c in numeric_cols if c not in [TARGET_COL, WEIGHT_COL, ID_COL, DATE_COL]]

high_corr_features = []

for col in numeric_cols:
    corr = train[col].corr(train[TARGET_COL])
    if pd.notna(corr) and abs(corr) > 0.85:
        high_corr_features.append((col, corr))

if high_corr_features:
    print(" Найдены признаки с высокой корреляцией:")
    for col, corr in sorted(high_corr_features, key=lambda x: abs(x[1]), reverse=True):
        print(f"      - {col}: {corr:.4f}")
else:
    print("Признаков с высокой корреляцией не найдено")


 Найдены признаки с высокой корреляцией:
      - first_salary_income: 0.9282
      - salary_6to12m_avg: 0.9277


## Проверка потенциальной утечки целевой переменной

Высокая корреляция с `target` используется как повод для проверки, но не как автоматическое доказательство leakage. Сильная связь возможна у корректного признака, например у исторической оценки зарплаты.

Для каждого подозрительного признака необходимо ответить на два вопроса:

1. Было ли значение доступно на дату `dt`?
2. Не рассчитано ли оно с использованием будущего периода или самой целевой переменной?

В текущем варианте новые признаки строятся только из исходных полей строки и не используют `target` или `w`.

### Отбор и исключение признаков

На этапе анализа данных и ручной проверки были выявлены признаки, которые не несут полезной информации для модели и могут быть безопасно удалены:

### 1. `first_salary_income`
- В тестовой выборке данный признак содержит **100% пропусков**
- Это делает его бесполезным для прогнозирования

### 2. `nonresident_flag`
Признак, указывающий на статус нерезидента РФ. При анализе было выявлено:

- Доля нерезидентов составляет всего **~1%** от всех наблюдений
- Средний доход между резидентами и нерезидентами **практически не отличается**
- Признак не даёт разделяющей способности

### 3. Региональные признаки
В данных присутствует несколько признаков, связанных с географией клиента:

| Признак | Описание | Решение |
|---------|----------|---------|
| `adminarea` | Регион клиента | **Оставляем** |
| `city_smart_name` | Город клиента | **Оставляем** |
| `addrref` | Регион отделения | **Удаляем** (дублирует `adminarea`) |
| `dp_address_unique_regions` | Количество уникальных регионов | **Удаляем** (дублирует информацию) |

**Решение:** оставляем только `adminarea` и `city_smart_name`, остальные удаляем.


## Проверенные гипотезы ручного исключения признаков

Закомментированные ячейки ниже отражают ручной анализ отдельных признаков:

- полностью или почти полностью пропущенные значения;
- редкие бинарные признаки;
- потенциально дублирующая региональная информация.

Эти варианты не удалены из notebook, потому что показывают ход исследования. При этом они не применяются в финальной конфигурации: локальные эксперименты не дали устойчивого улучшения WMAE либо основание для удаления оказалось недостаточно сильным.

Таким образом, окончательное решение консервативно: вручную не удалять признак только из-за редкости, пропусков или смысловой похожести без подтверждения качеством модели.

In [15]:
# # Проверяем first_salary_income
# if 'first_salary_income' in test.columns:
#     missing_pct = test['first_salary_income'].isna().mean() * 100
#     print(f"first_salary_income: {missing_pct:.1f}% пропусков в test")
#     missing_pct = train['first_salary_income'].isna().mean() * 100
#     print(f"first_salary_income: {missing_pct:.1f}% пропусков в train")

#     if missing_pct == 100:
#         print("Удаляем")
#         train = train.drop(columns=['first_salary_income'], errors='ignore')
#         test = test.drop(columns=['first_salary_income'], errors='ignore')

In [16]:
# # Проверяем nonresident_flag

# if 'nonresident_flag' in train.columns:
#     # 1. Распределение
#     print("\nРаспределение:")
#     display(train['nonresident_flag'].value_counts())
#     display(train['nonresident_flag'].value_counts(normalize=True))

#     # 2. Средний target по группам
#     print("\nСредний target по группам:")
#     display(train.groupby('nonresident_flag')[TARGET_COL].agg(['mean', 'count', 'std']))

#     # 3. Разница в target
#     resident_mean = train[train['nonresident_flag'] == 0][TARGET_COL].mean()
#     nonresident_mean = train[train['nonresident_flag'] == 1][TARGET_COL].mean()
#     diff = nonresident_mean - resident_mean

#     print(f"\nРазница в доходе между группами: {diff:.0f}")

#     if abs(diff) < 5000:
#         print("Разница минимальна, удаляем")
#         train = train.drop(columns=['nonresident_flag'], errors='ignore')
#         test = test.drop(columns=['nonresident_flag'], errors='ignore')
#     else:
#         print(f"Разница {diff:.0f}, оставляем")

In [17]:
# # Региональные признаки

# # Признаки, которые удаляем
# regions_to_drop = ['addrref', 'dp_address_unique_regions']

# train = train.drop(columns=[col], errors='ignore')
# test = test.drop(columns=[col], errors='ignore')

## Техническое исключение признаков

Автоматически удаляются только две категории признаков:

- **константные** — имеют одно и то же значение во всех строках и не способны разделять наблюдения;
- **точные дубликаты** — полностью совпадают и в train, и в test.

Сначала используется хеширование для быстрого поиска кандидатов, затем совпадение подтверждается прямым сравнением значений.

Сильно коррелирующие, но не идентичные признаки здесь не удаляются: для деревьев мультиколлинеарность менее критична, а похожие показатели могут содержать различия в отдельных сегментах.

In [18]:
protected_cols = {TARGET_COL, WEIGHT_COL, ID_COL, DATE_COL}
common_feature_cols = [
    c for c in train.columns
    if c in test.columns and c not in protected_cols
]

constant_cols = [
    c for c in common_feature_cols
    if train[c].nunique(dropna=False) <= 1
]

duplicate_of = {}
hash_groups = {}

for col in common_feature_cols:
    train_hash = int(pd.util.hash_pandas_object(train[col], index=False).sum())
    test_hash = int(pd.util.hash_pandas_object(test[col], index=False).sum())
    hash_groups.setdefault((train_hash, test_hash), []).append(col)

for group in hash_groups.values():
    if len(group) < 2:
        continue

    keep = group[0]
    for candidate in group[1:]:
        if train[keep].equals(train[candidate]) and test[keep].equals(test[candidate]):
            duplicate_of[candidate] = keep

duplicate_cols = sorted(duplicate_of)
technical_drop_cols = sorted(set(constant_cols + duplicate_cols))

print("Константные:", len(constant_cols))
print("Точные дубликаты:", len(duplicate_cols))
print("Всего безопасно удаляется:", len(technical_drop_cols))

train = train.drop(columns=technical_drop_cols, errors="ignore")
test = test.drop(columns=technical_drop_cols, errors="ignore")

Константные: 2
Точные дубликаты: 0
Всего безопасно удаляется: 2


## Формирование единой схемы данных

После обработки train и test приводятся к одному набору и порядку модельных признаков. В train дополнительно сохраняются `target` и `w`.

Контролируется, что:

- число строк не изменилось;
- набор признаков совпадает;
- порядок колонок совпадает;
- идентификаторы остались уникальными.

Эти проверки исключают тихие ошибки, при которых модель обучается на одном порядке признаков, а прогноз строится на другом.

In [19]:
feature_cols = [
    c for c in train.columns
    if c not in {TARGET_COL, WEIGHT_COL}
    and c in test.columns
]

train_output_cols = feature_cols + [TARGET_COL, WEIGHT_COL]
test_output_cols = feature_cols

train_processed = train[train_output_cols].copy()
test_processed = test[test_output_cols].copy()

if list(train_processed.drop(columns=[TARGET_COL, WEIGHT_COL]).columns) != list(test_processed.columns):
    raise AssertionError("Наборы или порядок признаков train/test различаются")

if len(train_processed) != len(train_raw):
    raise AssertionError("Изменилось число строк train")
if len(test_processed) != len(test_raw):
    raise AssertionError("Изменилось число строк test")

if train_processed[ID_COL].duplicated().any() or test_processed[ID_COL].duplicated().any():
    raise AssertionError("После обработки появились дубли ID")

print("Финальный train:", train_processed.shape)
print("Финальный test:", test_processed.shape)

Финальный train: (76786, 233)
Финальный test: (73214, 231)


## Аудит итоговых признаков

Для каждого оставшегося признака формируется технический профиль:

- тип данных в train и test;
- доля пропусков;
- число уникальных значений.

In [20]:
# Отчёт по признакам

audit_rows = []

for col in feature_cols:
    audit_rows.append({
        "feature": col,
        "dtype_train": str(train_processed[col].dtype),
        "dtype_test": str(test_processed[col].dtype),
        "missing_train_pct": train_processed[col].isna().mean() * 100,
        "missing_test_pct": test_processed[col].isna().mean() * 100,
        "nunique_train": train_processed[col].nunique(dropna=False),
        "nunique_test": test_processed[col].nunique(dropna=False)
    })

feature_audit = pd.DataFrame(audit_rows)

display(feature_audit.sort_values("missing_train_pct", ascending=False).head(30))

,feature,dtype_train,dtype_test,missing_train_pct,missing_test_pct,nunique_train,nunique_test
171,avg_by_category__amount__sum__cashflowcategory...,float64,float64,98.441122,98.403311,933,897
152,turn_fdep_db_avg_act_v2,float64,float64,96.972104,95.100664,2246,3508
79,turn_fdep_db_sum_v2,float64,float64,95.004298,93.170705,2248,3509
85,turn_fdep_db_avg_v2,float64,float64,95.004298,93.170705,2249,3509
150,avg_by_category__amount__sum__cashflowcategory...,float64,float64,93.199281,93.960172,2589,2197
128,avg_by_category__amount__sum__cashflowcategory...,float64,float64,92.748678,95.313738,3346,2078
119,dp_payoutincomedata_payout_avg_prev_year,float64,float64,89.749433,81.600514,7711,13069
220,first_salary_income,float64,float64,88.711484,100.000000,8669,1
100,avg_by_category__amount__sum__cashflowcategory...,float64,float64,88.460136,88.071953,4689,4520
174,by_category__amount__sum__eoperation_type_name...,float64,float64,88.167114,87.267463,4017,3645


In [21]:
train_processed.to_csv("train_processed_v2.csv", sep=";", decimal=",", index=False)
test_processed.to_csv("test_processed_v2.csv", sep=";", decimal=",", index=False)

In [22]:
# финальная проверка файлов.

train_check = pd.read_csv("train_processed_v2.csv", sep=";", decimal=",")
test_check = pd.read_csv("test_processed_v2.csv", sep=";", decimal=",")

assert train_check.shape == train_processed.shape
assert test_check.shape == test_processed.shape
assert list(train_check.drop(columns=[TARGET_COL, WEIGHT_COL]).columns) == list(test_check.columns)
assert train_check[ID_COL].is_unique
assert test_check[ID_COL].is_unique
assert train_check[TARGET_COL].notna().all()
assert train_check[WEIGHT_COL].notna().all()

print("Финальная проверка успешно пройдена.")

Финальная проверка успешно пройдена.
